# MaskGuard AI - Face Mask Detection System
**Computer Vision Portfolio Project**

---

## Setup Guide

### Step 1 - Runtime Configuration
Go to **Runtime -> Change runtime type -> T4 GPU** before running any cell.

### Step 2 - Dataset Setup
1. Visit https://www.kaggle.com/datasets/andrewmvd/face-mask-detection
2. Upload your `kaggle.json` API token when prompted in the dataset cell
3. The dataset is ~150MB

### Step 3 - Run All Cells in Order
Use **Runtime -> Run all** or execute cells sequentially.

### Step 4 - Launch the Gradio Interface
The final cell launches the interactive UI with a public shareable link.

### What's New (v3.0)
- **Gabor + HOG Fusion (Lab 07)** - 4-orientation Gabor filters capture mask weave texture that HOG misses, concatenated into a fused feature vector
- **CBIR Visual Retrieval Tab (Lab 08)** - Upload a face and see the top-3 most visually similar training examples that drove the prediction (Explainable AI)
- **PCA Pipeline (Labs 11+12)** - StandardScaler + PCA dimensionality reduction wrapping the SVM in a sklearn Pipeline for research-grade classification
- **Auto-Night Mode** - Gamma correction toggle that auto-brightens dark webcam feeds for better face detection in low-light conditions
- **Real-Time Webcam Streaming** - Live continuous detection, not single-frame capture
- **Multi-Cascade Face Detection** - 4 Haar/LBP cascades + CLAHE preprocessing for maximum face detection
- **Tunable Sensitivity** - Adjust detection sensitivity from the UI (no code changes needed)
- **Optimized Pipeline** - Faster inference for smoother streaming

### Important Notes
- Dataset contains ~853 annotated images in PASCAL VOC XML format
- Labels: `with_mask`, `without_mask`, `mask_weared_incorrect`
- HOG+Gabor+SVM baseline runs on CPU; MobileNetV2 fine-tuning requires GPU (T4)
- Webcam streaming in Gradio requires browser camera permissions
- If training loss stays high after epoch 5, verify the dataset path in the config cell

---

In [ ]:
!pip install "gradio>=4.44.0" opencv-python-headless torch torchvision pillow numpy matplotlib scikit-learn scikit-image tqdm kaggle lxml -q

In [ ]:
import os
import glob
import random
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from scipy.spatial.distance import cdist
from tqdm import tqdm
import gradio as gr
import warnings
import time
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU mode"}')
print(f'Gradio version: {gr.__version__}')

In [ ]:
from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle API configured.')

In [ ]:
!kaggle datasets download -d andrewmvd/face-mask-detection --unzip -p /content/facemask
print('Dataset downloaded.')
!ls /content/facemask

In [ ]:
IMAGES_DIR = '/content/facemask/images'
ANNOTS_DIR = '/content/facemask/annotations'

LABEL_MAP = {
    'with_mask': 0,
    'without_mask': 1,
    'mask_weared_incorrect': 2
}
LABEL_NAMES = ['With Mask', 'Without Mask', 'Incorrect Mask']
LABEL_COLORS = [(0, 220, 100), (0, 60, 255), (0, 165, 255)]

def parse_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bb = obj.find('bndbox')
        xmin = int(float(bb.find('xmin').text))
        ymin = int(float(bb.find('ymin').text))
        xmax = int(float(bb.find('xmax').text))
        ymax = int(float(bb.find('ymax').text))
        objects.append({'label': name, 'bbox': (xmin, ymin, xmax, ymax)})
    return filename, objects

all_annotations = {}
xml_files = glob.glob(os.path.join(ANNOTS_DIR, '*.xml'))

for xml_path in xml_files:
    filename, objects = parse_annotation(xml_path)
    all_annotations[filename] = objects

print(f'Annotations loaded: {len(all_annotations)} images')

label_counts = {'with_mask': 0, 'without_mask': 0, 'mask_weared_incorrect': 0}
for objs in all_annotations.values():
    for obj in objs:
        if obj['label'] in label_counts:
            label_counts[obj['label']] += 1

for k, v in label_counts.items():
    print(f'  {k}: {v}')

In [ ]:
def extract_face_patches(images_dir, annotations, target_size=(64, 64)):
    patches, labels, raw_labels = [], [], []
    for filename, objects in tqdm(annotations.items(), desc='Extracting patches'):
        img_path = os.path.join(images_dir, filename)
        if not os.path.exists(img_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        for obj in objects:
            label = obj['label']
            if label not in LABEL_MAP:
                continue
            xmin, ymin, xmax, ymax = obj['bbox']
            xmin, ymin = max(0, xmin), max(0, ymin)
            xmax = min(img.shape[1], xmax)
            ymax = min(img.shape[0], ymax)
            if xmax <= xmin or ymax <= ymin:
                continue
            patch = img[ymin:ymax, xmin:xmax]
            if patch.size == 0:
                continue
            patch_resized = cv2.resize(patch, target_size)
            patches.append(patch_resized)
            labels.append(LABEL_MAP[label])
            raw_labels.append(label)
    return patches, labels, raw_labels

all_patches, all_labels, all_raw = extract_face_patches(IMAGES_DIR, all_annotations)
print(f'Total face patches: {len(all_patches)}')

## Enhancement 01 — Gabor + HOG Fusion (Lab 07)
Gabor filters at 4 orientations capture mask weave texture patterns that HOG misses.
The 16-dim Gabor descriptor is concatenated with the HOG vector to form a fused feature.

In [ ]:
# gabor filters to catch mask weave textures
# 4 orientations x 4 frequencies = 16 Gabor kernels
# Each kernel produces mean + std energy -> 32-dim descriptor

def build_gabor_bank(num_orientations=4, num_frequencies=4):
    """Build a bank of Gabor filter kernels."""
    kernels = []
    for theta_idx in range(num_orientations):
        theta = theta_idx * np.pi / num_orientations
        for freq_idx in range(num_frequencies):
            frequency = 0.05 + freq_idx * 0.1
            sigma = 3.0
            lambd = 1.0 / frequency
            kernel_size = 9
            kernel = cv2.getGaborKernel(
                ksize=(kernel_size, kernel_size),
                sigma=sigma,
                theta=theta,
                lambd=lambd,
                gamma=0.5,
                psi=0
            )
            kernel = kernel / (1.5 * kernel.sum() + 1e-6)
            kernels.append(kernel)
    return kernels

gabor_kernels = build_gabor_bank()
print(f'Gabor filter bank created: {len(gabor_kernels)} kernels (4 orientations x 4 frequencies)')

def extract_gabor_features(gray_patch, kernels):
    """Extract Gabor energy features: mean + std for each kernel -> 32-dim vector."""
    features = []
    for kernel in kernels:
        filtered = cv2.filter2D(gray_patch, cv2.CV_64F, kernel)
        features.append(filtered.mean())
        features.append(filtered.std())
    return np.array(features, dtype=np.float64)

print(f'Gabor feature dimension: {len(gabor_kernels) * 2}')

In [ ]:
# extract and combine hog, gabor, AND color features
# HOG (grayscale edges) + Gabor (grayscale texture) + HSV (color histogram)
def extract_color_histogram(bgr_patch, bins=(8, 8, 8)):
    """Extract a normalized HSV color histogram."""
    hsv = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    return hist

def extract_hog_features_single(gray_patch):
    feat = hog(
        gray_patch, orientations=9, pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), visualize=False, feature_vector=True
    )
    return feat

def extract_fused_features(patches, kernels):
    all_feats = []
    for patch in tqdm(patches, desc='Extracting HOG+Gabor+Color features'):
        gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
        hog_feat = extract_hog_features_single(gray)
        gabor_feat = extract_gabor_features(gray, kernels)
        color_feat = extract_color_histogram(patch)
        fused = np.concatenate([hog_feat, gabor_feat, color_feat])
        all_feats.append(fused)
    return np.array(all_feats)

print('Extracting HOG + Gabor + Color fused features...')
fused_features = extract_fused_features(all_patches, gabor_kernels)
print(f'Fused feature vector shape: {fused_features.shape}')


## Enhancement 03 — PCA Pipeline (Labs 11 + 12)
- **StandardScaler** (Lab 11) normalizes HOG and Gabor features so dimensions don't fight each other
- **PCA** (Lab 12) compresses ~1796 dimensions -> 100 principal components, removing noise
- The SVM is wrapped in a **sklearn Pipeline** for research-grade classification

In [ ]:
# our research pipeline: standard scaler -> pca -> svm

X_train_fused, X_val_fused, y_train_fused, y_val_fused = train_test_split(
    fused_features, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

# Determine optimal PCA components (min of 100, num_samples, num_features)
n_components = min(100, X_train_fused.shape[0], X_train_fused.shape[1])
print(f'PCA components: {n_components}')

# Build the research-grade pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n_components, random_state=42)),
    # added class_weight='balanced' to stop the SVM from just guessing 'with_mask' for everything
    # since our dataset is heavily skewed (3232 masks vs 717 no-masks).
    ('svm', SVC(kernel='rbf', C=10, gamma='scale', probability=True, class_weight='balanced', random_state=42))
])

print('Training StandardScaler -> PCA -> SVM pipeline...')
svm_pipeline.fit(X_train_fused, y_train_fused)

pipeline_preds = svm_pipeline.predict(X_val_fused)
pipeline_acc = (pipeline_preds == np.array(y_val_fused)).mean() * 100

pca_model = svm_pipeline.named_steps['pca']
explained_var = np.sum(pca_model.explained_variance_ratio_) * 100

print(f'\nHOG+Gabor -> StandardScaler -> PCA -> SVM Validation Accuracy: {pipeline_acc:.2f}%')
print(f'PCA explained variance: {explained_var:.1f}% with {n_components} components')
print(f'\nClassification Report:')
print(classification_report(y_val_fused, pipeline_preds, target_names=LABEL_NAMES))

# Also keep a reference for backward compat
svm_clf = svm_pipeline

print('\nPipeline ready: StandardScaler -> PCA({}) -> SVM(rbf, C=10)'.format(n_components))

## Enhancement 02 — CBIR Visual Retrieval Tab (Lab 08)
Build a color histogram + Gabor feature index of every training patch.
Upload a face -> system shows the **top-3 most visually similar training examples**
that drove the prediction. **Explainable AI built entirely from classical CV.**

In [ ]:
# build the cbir index (color hist + gabor) for all training patches
# For each training patch: color histogram (HSV) + Gabor features


def extract_cbir_descriptor(bgr_patch, kernels):
    """Extract CBIR descriptor: color histogram + Gabor features."""
    color_hist = extract_color_histogram(bgr_patch)
    gray = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2GRAY)
    gabor_feat = extract_gabor_features(gray, kernels)
    return np.concatenate([color_hist, gabor_feat])

# Build CBIR index from ALL training patches
print('Building CBIR feature index for all patches...')

cbir_descriptors = []
cbir_patches = []
cbir_labels = []

for i, patch in enumerate(tqdm(all_patches, desc='Indexing CBIR features')):
    desc = extract_cbir_descriptor(patch, gabor_kernels)
    cbir_descriptors.append(desc)
    cbir_patches.append(patch.copy())
    cbir_labels.append(all_labels[i])

cbir_descriptors = np.array(cbir_descriptors)
cbir_labels = np.array(cbir_labels)

print(f'CBIR index built: {len(cbir_descriptors)} patches indexed')
print(f'CBIR descriptor dimension: {cbir_descriptors.shape[1]} (color hist: 512 + Gabor: {len(gabor_kernels)*2})')

In [ ]:
# cbir logic: finds the closest training examples to explain the prediction

def cbir_retrieve_top_k(query_bgr_patch, k=3):
    """
    Given a BGR face patch, retrieve the top-k most similar patches
    from the CBIR index using cosine distance.
    Returns: list of (patch_bgr, label_idx, distance) tuples
    """
    query_resized = cv2.resize(query_bgr_patch, (64, 64))
    query_desc = extract_cbir_descriptor(query_resized, gabor_kernels).reshape(1, -1)
    distances = cdist(query_desc, cbir_descriptors, metric='cosine')[0]
    top_k_indices = np.argsort(distances)[:k]
    results = []
    for idx in top_k_indices:
        results.append((
            cbir_patches[idx],
            int(cbir_labels[idx]),
            float(distances[idx])
        ))
    return results

def cbir_visual_explanation(input_image):
    """
    Gradio handler for the CBIR tab.
    Takes an uploaded face image, classifies it, and retrieves top-3 similar training patches.
    Returns: annotated result image, explanation text
    """
    if input_image is None:
        return None, 'No image provided.'

    if input_image.shape[-1] == 4:
        frame = cv2.cvtColor(input_image, cv2.COLOR_RGBA2BGR)
    elif input_image.shape[-1] == 3:
        frame = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)
    else:
        return None, 'Invalid image format.'

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face_boxes = detect_faces(gray, sensitivity=1.0)

    if not face_boxes:
        face_patch = cv2.resize(frame, (64, 64))
    else:
        x1, y1, x2, y2 = face_boxes[0]
        face_patch = frame[y1:y2, x1:x2]
        if face_patch.size == 0:
            face_patch = cv2.resize(frame, (64, 64))
        else:
            face_patch = cv2.resize(face_patch, (64, 64))

    gray_face = cv2.cvtColor(face_patch, cv2.COLOR_BGR2GRAY)
    hog_feat = extract_hog_features_single(gray_face)
    gabor_feat = extract_gabor_features(gray_face, gabor_kernels)
    color_feat = extract_color_histogram(face_patch)
    fused_feat = np.concatenate([hog_feat, gabor_feat, color_feat]).reshape(1, -1)
    prediction = svm_pipeline.predict(fused_feat)[0]
    proba = svm_pipeline.predict_proba(fused_feat)[0]

    top3 = cbir_retrieve_top_k(face_patch, k=3)

    cell_size = 150
    padding = 10
    canvas_w = (cell_size + padding) * 4 + padding
    canvas_h = cell_size * 2 + padding * 3 + 60
    canvas = np.full((canvas_h, canvas_w, 3), (12, 18, 7), dtype=np.uint8)

    query_display = cv2.resize(face_patch, (cell_size, cell_size))
    y_off = padding + 30
    x_off = padding
    canvas[y_off:y_off+cell_size, x_off:x_off+cell_size] = query_display
    cv2.putText(canvas, 'YOUR QUERY', (x_off, y_off - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 229, 204), 1)

    pred_label = LABEL_NAMES[prediction]
    pred_conf = proba[prediction] * 100
    pred_color = LABEL_COLORS[prediction]
    cv2.putText(canvas, f'{pred_label}', (x_off, y_off + cell_size + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, pred_color, 1)
    cv2.putText(canvas, f'{pred_conf:.1f}% conf', (x_off, y_off + cell_size + 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180, 180, 180), 1)

    for i, (patch_bgr, lbl_idx, dist) in enumerate(top3):
        x_pos = padding + (i + 1) * (cell_size + padding)
        patch_display = cv2.resize(patch_bgr, (cell_size, cell_size))
        border_color = LABEL_COLORS[lbl_idx]
        cv2.rectangle(canvas, (x_pos - 2, y_off - 2),
                      (x_pos + cell_size + 2, y_off + cell_size + 2),
                      border_color, 2)
        canvas[y_off:y_off+cell_size, x_pos:x_pos+cell_size] = patch_display
        rank_text = f'#{i+1} Match'
        cv2.putText(canvas, rank_text, (x_pos, y_off - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 229, 204), 1)
        cv2.putText(canvas, LABEL_NAMES[lbl_idx], (x_pos, y_off + cell_size + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, border_color, 1)
        sim_pct = max(0, (1.0 - dist)) * 100
        cv2.putText(canvas, f'Sim: {sim_pct:.0f}%', (x_pos, y_off + cell_size + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (140, 140, 140), 1)

    canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

    explanation_lines = [
        f'Prediction: {pred_label} ({pred_conf:.1f}% confidence)',
        f'',
        f'--- Top-3 Most Similar Training Examples ---',
    ]
    for i, (_, lbl_idx, dist) in enumerate(top3):
        sim_pct = max(0, (1.0 - dist)) * 100
        explanation_lines.append(
            f'  #{i+1}: {LABEL_NAMES[lbl_idx]} (similarity: {sim_pct:.1f}%)'
        )
    explanation_lines.append('')
    explanation_lines.append('The system found these training patches most similar')
    explanation_lines.append('to your query using Color Histogram + Gabor texture')
    explanation_lines.append('features (CBIR - Content-Based Image Retrieval).')
    explanation_lines.append(f'\nClassifier: HOG+Gabor -> StandardScaler -> PCA -> SVM')

    matching = sum(1 for _, lbl, _ in top3 if lbl == prediction)
    if matching == 3:
        explanation_lines.append('\nAll 3 similar examples agree with prediction!')
    elif matching >= 2:
        explanation_lines.append(f'\n{matching}/3 similar examples support the prediction.')
    else:
        explanation_lines.append(f'\nOnly {matching}/3 similar examples match - prediction may be uncertain.')

    return canvas_rgb, '\n'.join(explanation_lines)

print('CBIR retrieval functions ready.')

In [ ]:
def apply_morphological_ops(face_patch):
    hsv = cv2.cvtColor(face_patch, cv2.COLOR_BGR2HSV)
    lower_mask = np.array([0, 0, 150])
    upper_mask = np.array([180, 80, 255])
    mask_region = cv2.inRange(hsv, lower_mask, upper_mask)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_region = cv2.morphologyEx(mask_region, cv2.MORPH_CLOSE, kernel)
    mask_region = cv2.morphologyEx(mask_region, cv2.MORPH_OPEN, kernel)
    mask_region = cv2.dilate(mask_region, kernel, iterations=2)
    contours, _ = cv2.findContours(mask_region, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    vis = face_patch.copy()
    cv2.drawContours(vis, contours, -1, (0, 255, 200), 2)
    return mask_region, contours, vis

print('Morphological processing functions ready.')

In [ ]:
class MaskDataset(Dataset):
    def __init__(self, patches, labels, transform=None):
        self.patches = patches
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        patch = self.patches[idx]
        img = Image.fromarray(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

cnn_patches, cnn_labels = [], []
for patch, label in zip(all_patches, all_labels):
    p128 = cv2.resize(patch, (128, 128))
    cnn_patches.append(p128)
    cnn_labels.append(label)

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

X_tr, X_va, y_tr, y_va = train_test_split(
    cnn_patches, cnn_labels, test_size=0.2, random_state=42, stratify=cnn_labels
)

train_ds = MaskDataset(X_tr, y_tr, train_transform)
val_ds = MaskDataset(X_va, y_va, val_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')

In [ ]:
mobilenet = models.mobilenet_v2(pretrained=True)

for param in mobilenet.features[:14].parameters():
    param.requires_grad = False

mobilenet.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(mobilenet.last_channel, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(256, 3)
)

mobilenet = mobilenet.to(DEVICE)

class_weights = torch.tensor([1.0, 2.5, 2.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, mobilenet.parameters()), lr=5e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

trainable = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
total = sum(p.numel() for p in mobilenet.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

In [ ]:
EPOCHS = 20
best_val_acc = 0.0
train_losses, val_losses, val_accs = [], [], []

for epoch in range(EPOCHS):
    mobilenet.train()
    running_loss = 0.0
    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        outputs = mobilenet(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    mobilenet.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs = mobilenet(imgs)
            val_loss += criterion(outputs, lbls).item()
            _, predicted = torch.max(outputs, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    accuracy = correct / total * 100
    val_losses.append(avg_val_loss)
    val_accs.append(accuracy)
    scheduler.step()

    if accuracy > best_val_acc:
        best_val_acc = accuracy
        torch.save(mobilenet.state_dict(), '/content/mask_mobilenet_best.pth')

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {accuracy:.2f}% | Best: {best_val_acc:.2f}%')

mobilenet.load_state_dict(torch.load('/content/mask_mobilenet_best.pth'))
print(f'Best model loaded. Best Val Accuracy: {best_val_acc:.2f}%')

## Face Detection
Uses OpenCV Haar Cascades with CLAHE preprocessing. We dropped the multi-cascade overkill from v2 because it was tanking the webcam framerate and causing false positives on background objects.

In [ ]:
# simplified face detection. running 4 cascades on 3 contrast variations was killing webcam fps
# and causing weird background artifacts (like pillars) to get detected as faces.
cascade_alt2 = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')
cascade_default = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def detect_faces(gray, sensitivity=1.0):
    gray_clahe = clahe.apply(gray)
    
    # dial back the parameters so we don't pick up garbage in the background
    min_neighbors = max(3, int(7 - sensitivity * 2))
    min_face_size = max(30, int(70 - sensitivity * 20))
    scale_factor = max(1.05, 1.15 - (sensitivity * 0.05))
    
    faces = cascade_alt2.detectMultiScale(
        gray_clahe, scaleFactor=scale_factor, minNeighbors=min_neighbors, minSize=(min_face_size, min_face_size)
    )
    
    # fallback to default cascade if we missed the face entirely
    if len(faces) == 0:
        faces = cascade_default.detectMultiScale(
            gray_clahe, scaleFactor=scale_factor, minNeighbors=min_neighbors, minSize=(min_face_size, min_face_size)
        )
        
    return [(int(x), int(y), int(x+w), int(y+h)) for (x, y, w, h) in faces]

print('Face detection ready (optimized to reduce false positives)')


## Auto-Night Mode — Gamma Correction
Measures mean brightness of each webcam frame. If below threshold (or always when toggled on),
applies **adaptive gamma correction** via a LUT (lookup table) to brighten dark feeds before face detection.
Gamma auto-adjusts based on how dark the frame is (darker = stronger correction).

In [ ]:
# auto-night mode: bumps up gamma for dark webcam feeds

NIGHT_MODE_BRIGHTNESS_THRESHOLD = 80  # mean pixel value below this = dark frame

def apply_gamma_correction(frame_bgr, auto=True):
    """
    Apply adaptive gamma correction to brighten dark frames.
    If auto=True, gamma is computed from mean brightness.
    Frames above the brightness threshold are returned unchanged.
    Returns: (corrected_frame, was_corrected, brightness, gamma_used)
    """
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    mean_brightness = float(gray.mean())

    if auto and mean_brightness >= NIGHT_MODE_BRIGHTNESS_THRESHOLD:
        # Frame is bright enough, no correction needed
        return frame_bgr, False, mean_brightness, 1.0

    # Auto-calculate gamma: darker frames get stronger correction
    # Maps brightness 0-80 to gamma 2.5-1.2 (darker = higher gamma)
    ratio = mean_brightness / NIGHT_MODE_BRIGHTNESS_THRESHOLD
    gamma = float(np.clip(2.5 - ratio * 1.3, 1.1, 2.5))

    # Build lookup table for fast gamma correction
    inv_gamma = 1.0 / gamma
    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255 for i in range(256)
    ]).astype('uint8')
    corrected = cv2.LUT(frame_bgr, table)

    return corrected, True, mean_brightness, gamma

# Quick test
dark_test = np.full((100, 100, 3), 40, dtype=np.uint8)
bright_test = np.full((100, 100, 3), 150, dtype=np.uint8)
_, dark_corrected, dark_br, dark_g = apply_gamma_correction(dark_test)
_, bright_corrected, bright_br, bright_g = apply_gamma_correction(bright_test)
print(f'Dark frame test:   brightness={dark_br:.0f} -> corrected={dark_corrected}, gamma={dark_g:.2f}')
print(f'Bright frame test: brightness={bright_br:.0f} -> corrected={bright_corrected}, gamma={bright_g:.2f}')
print('Auto-Night Mode (Gamma Correction) ready.')

In [ ]:
# Pre-create the CNN transform for inference (avoid recreating every frame)
cnn_inference_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def detect_and_classify(frame_bgr, use_svm_baseline=False, sensitivity=1.0):
    """
    Detect faces and classify mask status.
    Uses multi-cascade detection with adjustable sensitivity.
    SVM baseline now uses HOG+Gabor fused features through the PCA pipeline.
    """
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    face_boxes = detect_faces(gray, sensitivity=sensitivity)

    results = []
    for (x1, y1, x2, y2) in face_boxes:
        w = x2 - x1
        h = y2 - y1
        margin_x = int(0.1 * w)
        margin_y = int(0.1 * h)
        fx1 = max(0, x1 - margin_x)
        fy1 = max(0, y1 - margin_y)
        fx2 = min(frame_bgr.shape[1], x2 + margin_x)
        fy2 = min(frame_bgr.shape[0], y2 + margin_y)
        face_patch = frame_bgr[fy1:fy2, fx1:fx2]
        if face_patch.size == 0:
            continue
        face_resized = cv2.resize(face_patch, (128, 128))

        if use_svm_baseline:
            face_svm = cv2.resize(face_patch, (64, 64))
            gray_face = cv2.cvtColor(face_svm, cv2.COLOR_BGR2GRAY)
            hog_feat = extract_hog_features_single(gray_face)
            gabor_feat = extract_gabor_features(gray_face, gabor_kernels)
            color_feat = extract_color_histogram(face_svm)
            fused_feat = np.concatenate([hog_feat, gabor_feat, color_feat]).reshape(1, -1)
            proba = svm_pipeline.predict_proba(fused_feat)[0]
            pred_idx = np.argmax(proba)
            conf = proba[pred_idx]
        else:
            img_pil = Image.fromarray(cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB))
            tensor = cnn_inference_transform(img_pil).unsqueeze(0).to(DEVICE)
            mobilenet.eval()
            with torch.no_grad():
                output = mobilenet(tensor)
                probs = torch.softmax(output, dim=1).cpu().numpy()[0]
            pred_idx = np.argmax(probs)
            conf = probs[pred_idx]

        morph_mask, contours, morph_vis = apply_morphological_ops(face_resized)
        mask_coverage = np.sum(morph_mask > 0) / morph_mask.size * 100

        results.append({
            'bbox': (fx1, fy1, fx2, fy2),
            'label_idx': pred_idx,
            'label': LABEL_NAMES[pred_idx],
            'conf': conf,
            'morph_vis': morph_vis,
            'mask_coverage': mask_coverage
        })

    return results, len(face_boxes)

def draw_detections(frame_bgr, results, show_morph=False):
    output = frame_bgr.copy()
    h, w = output.shape[:2]

    for res in results:
        x1, y1, x2, y2 = res['bbox']
        color = LABEL_COLORS[res['label_idx']]
        cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

        label_txt = f"{res['label']} {res['conf']*100:.1f}%"
        (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(output, (x1, y1 - th - 10), (x1 + tw + 8, y1), color, -1)
        cv2.putText(output, label_txt, (x1 + 4, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

        cov_txt = f"Cov: {res['mask_coverage']:.1f}%"
        cv2.putText(output, cov_txt, (x1 + 4, y2 + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    cv2.rectangle(output, (0, 0), (w, 70), (8, 8, 12), -1)
    cv2.putText(output, 'MASKGUARD AI', (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 220, 180), 2)
    with_count = sum(1 for r in results if r['label_idx'] == 0)
    without_count = sum(1 for r in results if r['label_idx'] == 1)
    wrong_count = sum(1 for r in results if r['label_idx'] == 2)
    cv2.putText(output, f'Faces: {len(results)}  Masked: {with_count}  Unmasked: {without_count}  Wrong: {wrong_count}',
                (12, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 180, 180), 1)

    return output

print('Detection and drawing functions ready.')

In [ ]:
def gradio_detect_frame(frame_bgr, model_choice, show_morph, sensitivity):
    if frame_bgr is None:
        return None, None, 'No frame provided.'
    frame = cv2.resize(frame_bgr, (640, 480))
    use_svm = (model_choice == 'HOG+Gabor+PCA+SVM (Enhanced)')
    results, face_count = detect_and_classify(frame, use_svm_baseline=use_svm, sensitivity=sensitivity)
    output_frame = draw_detections(frame, results, show_morph)
    morph_panels = []
    if show_morph and results:
        for res in results[:4]:
            morph_panels.append(res['morph_vis'])
    morph_grid = None
    if morph_panels:
        while len(morph_panels) < 4:
            morph_panels.append(np.zeros((128, 128, 3), dtype=np.uint8))
        top = np.hstack(morph_panels[:2])
        bot = np.hstack(morph_panels[2:4])
        morph_grid = np.vstack([top, bot])
        morph_grid = cv2.cvtColor(morph_grid, cv2.COLOR_BGR2RGB)
    stats_lines = [f'Total Faces Detected: {face_count}']
    stats_lines.append(f'Detection Sensitivity: {sensitivity:.1f}')
    if results:
        for i, res in enumerate(results):
            stats_lines.append(
                f'Face {i+1}: {res["label"]} - {res["conf"]*100:.1f}% | Coverage: {res["mask_coverage"]:.1f}%'
            )
        with_mask = sum(1 for r in results if r['label_idx'] == 0)
        compliance = with_mask / len(results) * 100
        stats_lines.append(f'\nMask Compliance Rate: {compliance:.1f}%')
        stats_lines.append(f'Model: {model_choice}')
    else:
        stats_lines.append('\nNo faces detected.')
        stats_lines.append('Try increasing the Detection Sensitivity slider!')
    output_rgb = cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB)
    return output_rgb, morph_grid, '\n'.join(stats_lines)

def gradio_detect_image(input_image, model_choice, show_morph, sensitivity):
    if input_image is None:
        return None, None, 'No image provided.'
    if input_image.shape[-1] == 4:
        frame = cv2.cvtColor(input_image, cv2.COLOR_RGBA2BGR)
    elif input_image.shape[-1] == 3:
        frame = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)
    else:
        return None, None, 'Invalid image format.'
    return gradio_detect_frame(frame, model_choice, show_morph, sensitivity)

# webcam streaming handler
latest_webcam_stats = 'Starting webcam...'

def webcam_stream_with_stats(webcam_frame, model_choice, sensitivity, night_mode):
    """Streaming handler: processes every webcam frame and updates stats.
    If night_mode is enabled, applies adaptive gamma correction to dark frames."""
    global latest_webcam_stats
    if webcam_frame is None:
        return None, 'Waiting for webcam...'

    if webcam_frame.shape[-1] == 4:
        frame = cv2.cvtColor(webcam_frame, cv2.COLOR_RGBA2BGR)
    else:
        frame = cv2.cvtColor(webcam_frame, cv2.COLOR_RGB2BGR)

    frame = cv2.resize(frame, (640, 480))

    # Auto-Night Mode: apply gamma correction if toggle is on
    night_applied = False
    brightness_val = 0.0
    gamma_val = 1.0
    if night_mode:
        frame, night_applied, brightness_val, gamma_val = apply_gamma_correction(frame, auto=True)

    use_svm = (model_choice == 'HOG+Gabor+PCA+SVM (Enhanced)')
    results, face_count = detect_and_classify(frame, use_svm_baseline=use_svm, sensitivity=sensitivity)

    if results:
        annotated = draw_detections(frame, results)
        lines = [f'Faces: {face_count}']
        for i, r in enumerate(results):
            lines.append(f'  #{i+1}: {r["label"]} ({r["conf"]*100:.0f}%)')
        with_mask = sum(1 for r in results if r['label_idx'] == 0)
        compliance = with_mask / len(results) * 100
        lines.append(f'Compliance: {compliance:.0f}%')
        if night_mode:
            if night_applied:
                lines.append(f'Night Mode: ON (gamma={gamma_val:.2f})')
            else:
                lines.append(f'Night Mode: ON (bright enough, no correction)')
            lines.append(f'Brightness: {brightness_val:.0f}/255')
        latest_webcam_stats = '\n'.join(lines)
    else:
        h, w = frame.shape[:2]
        annotated = frame.copy()
        cv2.rectangle(annotated, (0, 0), (w, 70), (8, 8, 12), -1)
        cv2.putText(annotated, 'MASKGUARD AI - SCANNING...', (12, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)
        cv2.putText(annotated, 'No face detected', (12, 58),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (120, 120, 120), 1)
        night_info = ''
        if night_mode:
            if night_applied:
                night_info = f'\nNight Mode: ON (gamma={gamma_val:.2f}) | Brightness: {brightness_val:.0f}/255'
            else:
                night_info = f'\nNight Mode: ON (bright enough) | Brightness: {brightness_val:.0f}/255'
        latest_webcam_stats = f'No face detected\nIncrease sensitivity slider!{night_info}'

    output_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    return output_rgb, latest_webcam_stats

# video file processor
def gradio_detect_video(input_video_path, model_choice, frame_skip):
    if input_video_path is None:
        return None, 'No video uploaded.'
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        return None, 'Could not open video file.'
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_path = '/content/maskguard_output.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = None
    frame_idx = 0
    processed = 0
    total_faces = 0
    total_masked = 0
    skip = max(1, int(frame_skip))
    use_svm = (model_choice == 'HOG+Gabor+PCA+SVM (Enhanced)')
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if frame_idx % skip != 0:
            frame_idx += 1
            continue
        frame_bgr = cv2.resize(frame_bgr, (640, 480))
        results, face_count = detect_and_classify(frame_bgr, use_svm_baseline=use_svm, sensitivity=1.0)
        annotated = draw_detections(frame_bgr, results)
        if writer is None:
            h, w = annotated.shape[:2]
            writer = cv2.VideoWriter(out_path, fourcc, fps / skip, (w, h))
        writer.write(annotated)
        total_faces += face_count
        total_masked += sum(1 for r in results if r['label_idx'] == 0)
        processed += 1
        frame_idx += 1
    cap.release()
    if writer:
        writer.release()
    avg_faces = total_faces / processed if processed > 0 else 0
    overall_compliance = total_masked / total_faces * 100 if total_faces > 0 else 0
    summary = (
        f'Total Input Frames: {total_frames}\n'
        f'Frames Processed: {processed} (every {skip} frame(s))\n'
        f'Total Faces Detected: {total_faces}\n'
        f'Avg Faces Per Frame: {avg_faces:.1f}\n'
        f'Overall Mask Compliance: {overall_compliance:.1f}%\n'
        f'Model Used: {model_choice}\n'
        f'Output saved to: {out_path}'
    )
    return out_path, summary

print('All inference functions ready (including real-time webcam streaming + night mode).')

## GRADIO INTERFACE - Real-Time Webcam + Image + Video + CBIR Retrieval

In [ ]:
MODEL_CHOICES = ['MobileNetV2 (CNN)', 'HOG+Gabor+PCA+SVM (Enhanced)']

TEAL_THEME = gr.themes.Base(primary_hue='teal', secondary_hue='cyan', neutral_hue='slate', font=[gr.themes.GoogleFont('DM Mono'), 'monospace']).set(body_background_fill='#040d0f', block_background_fill='#071214', block_border_color='#0d3338', input_background_fill='#0a1a1d', button_primary_background_fill='#007a7a', button_primary_background_fill_hover='#009999')

custom_css = """#title-block {
    background: linear-gradient(135deg, #040d0f 0%, #071a1d 50%, #040d0f 100%);
    border: 1px solid #009999;
    border-radius: 12px;
    padding: 24px;
    margin-bottom: 16px;
    text-align: center;
}
#title-block h1 {
    font-family: 'DM Mono', monospace;
    font-size: 2.2rem;
    color: #00e5cc;
    letter-spacing: 4px;
    text-transform: uppercase;
    text-shadow: 0 0 20px rgba(0, 229, 204, 0.45);
}
#title-block p {
    color: #4a9a9a;
    font-family: 'DM Mono', monospace;
    font-size: 0.85rem;
    letter-spacing: 2px;
}
.stats-box {
    background: #071214;
    border: 1px solid #0d3338;
    border-radius: 8px;
    font-family: 'DM Mono', monospace;
    color: #00e5cc;
}
.gradio-container { max-width: 1400px !important; }
.live-badge {
    display: inline-block;
    background: #ff3333;
    color: white;
    padding: 2px 10px;
    border-radius: 12px;
    font-family: 'DM Mono', monospace;
    font-size: 0.75rem;
    font-weight: bold;
    animation: pulse 1.5s ease-in-out infinite;
}
@keyframes pulse {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.5; }
}
.cbir-info {
    background: linear-gradient(135deg, #0a1a1d, #071214);
    border: 1px solid #0d3338;
    border-radius: 8px;
    padding: 12px 16px;
    font-family: 'DM Mono', monospace;
    font-size: 0.82rem;
    color: #4a9a9a;
    margin-bottom: 12px;
}"""

with gr.Blocks(theme=TEAL_THEME, css=custom_css, title='MaskGuard AI') as demo:
    with gr.Column(elem_id='title-block'):
        gr.HTML('<h1>MASKGUARD AI</h1><p>Real-Time Face Mask Detection &mdash; HOG+Gabor Fusion &middot; PCA Pipeline &middot; CBIR Explainability &middot; Night Mode &middot; MobileNetV2</p>')
    with gr.Tabs():
        with gr.Tab('Webcam (Real-Time)'):
            gr.HTML('<div style="padding:12px 16px;background:#071214;border:1px solid #0d3338;border-radius:8px;margin-bottom:12px;font-family:DM Mono,monospace;font-size:0.85rem;color:#4a9a9a"><span class="live-badge">LIVE</span> &nbsp; Real-time streaming detection. Grant camera permission when prompted. Enable <strong>Auto-Night Mode</strong> if your room is dark!</div>')
            with gr.Row():
                with gr.Column(scale=2):
                    webcam_stream = gr.Image(label='Live Webcam Feed', sources=['webcam'], streaming=True, type='numpy', height=480, show_label=True)
                    webcam_output = gr.Image(label='Detection Output (Real-Time)', height=480, show_label=True)
                with gr.Column(scale=1):
                    gr.Markdown('### Controls')
                    cam_model = gr.Radio(choices=MODEL_CHOICES, value='MobileNetV2 (CNN)', label='Classifier Model')
                    cam_sensitivity = gr.Slider(minimum=0.0, maximum=2.0, step=0.1, value=1.0, label='Detection Sensitivity')
                    cam_night_mode = gr.Checkbox(label='Auto-Night Mode (Gamma Correction)', value=False)
                    gr.Markdown('### Live Stats')
                    cam_stats = gr.Textbox(label='', lines=10, interactive=False, elem_classes='stats-box', value='Waiting for webcam...')
            webcam_stream.stream(fn=webcam_stream_with_stats, inputs=[webcam_stream, cam_model, cam_sensitivity, cam_night_mode], outputs=[webcam_output, cam_stats], stream_every=0.1)
        with gr.Tab('Image Detection'):
            with gr.Row():
                with gr.Column(scale=1):
                    input_img = gr.Image(label='Upload Image', type='numpy', height=280)
                    img_model = gr.Radio(choices=MODEL_CHOICES, value='MobileNetV2 (CNN)', label='Classifier')
                    img_sensitivity = gr.Slider(minimum=0.0, maximum=2.0, step=0.1, value=1.0, label='Detection Sensitivity')
                    img_show_morph = gr.Checkbox(label='Show Morphological Analysis', value=True)
                    img_run_btn = gr.Button('DETECT MASKS', variant='primary')
                    img_stats = gr.Textbox(label='Report', lines=8, interactive=False, elem_classes='stats-box')
                with gr.Column(scale=2):
                    img_detection_out = gr.Image(label='Detection Result', height=400)
                    img_morph_out = gr.Image(label='Morphology', height=260)
            img_run_btn.click(fn=gradio_detect_image, inputs=[input_img, img_model, img_show_morph, img_sensitivity], outputs=[img_detection_out, img_morph_out, img_stats])
        with gr.Tab('Video Detection'):
            with gr.Row():
                with gr.Column(scale=1):
                    input_video = gr.Video(label='Upload Video')
                    vid_model = gr.Radio(choices=MODEL_CHOICES, value='MobileNetV2 (CNN)', label='Classifier')
                    frame_skip = gr.Slider(minimum=1, maximum=5, step=1, value=2, label='Frame Skip')
                    vid_run_btn = gr.Button('PROCESS VIDEO', variant='primary')
                    vid_stats = gr.Textbox(label='Summary', lines=8, interactive=False, elem_classes='stats-box')
                with gr.Column(scale=2):
                    vid_output = gr.Video(label='Annotated Video')
            vid_run_btn.click(fn=gradio_detect_video, inputs=[input_video, vid_model, frame_skip], outputs=[vid_output, vid_stats])
        with gr.Tab('Visual Retrieval (CBIR)'):
            gr.HTML('<div class="cbir-info">Explainable AI via CBIR &mdash; Upload a face image and see the top-3 most visually similar training examples that drove the prediction. Uses Color Histogram + Gabor texture features (Lab 08).</div>')
            with gr.Row():
                with gr.Column(scale=1):
                    cbir_input = gr.Image(label='Upload a Face Image', type='numpy', height=300)
                    cbir_run_btn = gr.Button('FIND SIMILAR AND EXPLAIN', variant='primary')
                    cbir_explanation = gr.Textbox(label='Explanation', lines=12, interactive=False, elem_classes='stats-box')
                with gr.Column(scale=2):
                    cbir_result = gr.Image(label='Query + Top-3 Similar Training Examples', height=420)
            cbir_run_btn.click(fn=cbir_visual_explanation, inputs=[cbir_input], outputs=[cbir_result, cbir_explanation])

demo.launch(share=True)
print('MaskGuard AI is live!')